# ส่วนที่ 2 · Extractive: TextRank / TF-IDF เลือกประโยคสำคัญ

**ผู้รับผิดชอบ:** คนที่ 2 (คู่ B) · **ส่ง:** ตาราง ROUGE + `predictions_textrank.csv`

**วิธีใช้:** Runtime → **Run all** (ไม่ต้องใช้ GPU)

- ข้อมูล: [ThaiSum](https://huggingface.co/datasets/pythainlp/thaisum) โหลดแบบ streaming เหมือน `04-mt5.ipynb` · **test ชุดเดียวกัน** (500 ข่าวแรกของ test) เพื่อเทียบกับ mT5/mBART ได้ตรงกัน
- วิธี extractive ไม่สร้างคำใหม่ แต่ **เลือกประโยคจากตัวข่าว** มาต่อกันเป็นบทสรุป เทียบ 3 วิธี:
  1. **Lead-k** (baseline): เอา k ประโยคแรกของข่าว
  2. **TF-IDF**: ให้คะแนนประโยคจากน้ำหนัก TF-IDF ของคำในประโยค
  3. **TextRank**: สร้างกราฟประโยค (ความคล้ายแบบ cosine ของ TF-IDF) แล้วรัน PageRank
- เลือกวิธีและค่า k ที่ดีที่สุดจาก **validation** แล้วค่อยวัดบน **test** ครั้งเดียว
- ถ้าคนที่ 1 แจกไฟล์ train/val/test แล้ว ให้ตั้ง `TRAIN_CSV` / `VAL_CSV` / `TEST_CSV` (คอลัมน์ `body`, `summary`)

In [ ]:
!pip -q install datasets pythainlp rouge_score scikit-learn python-crfsuite

## 1. ตั้งค่า (แก้แค่ช่องนี้)

In [ ]:
# ข้อมูล: None = โหลด ThaiSum จาก Hugging Face · ใส่ path = ใช้ไฟล์ของคนที่ 1
TRAIN_CSV, VAL_CSV, TEST_CSV = None, None, None
IDF_SIZE  = 1000     # จำนวนข่าวที่ใช้คำนวณ IDF (ไม่ใช่การเทรนโมเดล ใช้ train split เท่านั้น)
VAL_SIZE  = 300      # ใช้เลือกวิธี/ค่า k
TEST_SIZE = 500      # ทุกโมเดลต้องใช้ test ชุดเดียวกัน

SENT_ENGINE = "crfcut"            # ตัดประโยค: "crfcut" (โมเดล CRF ของ PyThaiNLP) หรือ "space" (ตัดตามช่องว่าง)
MIN_LEN = 15                      # ประโยคสั้นกว่านี้ (ตัวอักษร) จะถูกรวมกับประโยคถัดไป
K_GRID  = [1, 2, 3]               # จำนวนประโยคที่เลือกเป็นบทสรุป
METHODS = ["lead", "tfidf", "textrank"]
OUTPUT_NAME = "textrank"          # ชื่อไฟล์ผลลัพธ์
SEED = 42

## 2. โหลดข้อมูล (โค้ดเดียวกับ `04-mt5.ipynb`)

In [ ]:
import pandas as pd
from datasets import load_dataset

def from_thaisum(split, n, shuffle=False):
    ds = load_dataset("pythainlp/thaisum", split=split, streaming=True)
    if shuffle:
        ds = ds.shuffle(seed=SEED, buffer_size=10_000)
    return pd.DataFrame(list(ds.take(n)))[["body", "summary"]]

def load(csv, split, n, shuffle=False):
    df = pd.read_csv(csv) if csv else from_thaisum(split, n, shuffle)
    df = df.dropna(subset=["body", "summary"])
    return df[(df.body.str.len() > 0) & (df.summary.str.len() > 0)].reset_index(drop=True)

idf_df  = load(TRAIN_CSV, "train", IDF_SIZE, shuffle=True)
val_df  = load(VAL_CSV, "validation", VAL_SIZE)
test_df = load(TEST_CSV, "test", TEST_SIZE)
print("idf", len(idf_df), "| val", len(val_df), "| test", len(test_df))
print("\nตัวอย่างข่าว:", val_df.body[0][:200], "...\nบทสรุป:", val_df.summary[0])

## 3. ROUGE สำหรับภาษาไทย (ฟังก์ชันเดียวกับทุกคน)

In [ ]:
from rouge_score import rouge_scorer
from pythainlp.tokenize import word_tokenize

class ThaiTokenizer:
    def tokenize(self, text):
        return [w for w in word_tokenize(text, engine="newmm") if w.strip()]

scorer = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], tokenizer=ThaiTokenizer())

def thai_rouge(preds, refs):
    s = [scorer.score(r, p) for p, r in zip(preds, refs)]
    return {k: round(100 * sum(x[k].fmeasure for x in s) / len(s), 2) for k in ["rouge1", "rouge2", "rougeL"]}

print(thai_rouge(["นายกรัฐมนตรีประชุมครม.วันนี้"], ["วันนี้นายกรัฐมนตรีเป็นประธานประชุมครม."]))

## 4. ตัดประโยค + TF-IDF
ภาษาไทยไม่มีจุดจบประโยคชัดเจน จึงตัดบรรทัดก่อน แล้วใช้ `crfcut` ตัดต่อ และรวมประโยคที่สั้นเกินไปเข้าด้วยกัน
คำหยุด (stopword) ถูกตัดออกก่อนคิด TF-IDF ส่วนค่า IDF คำนวณจากประโยคของข่าวใน train split

In [ ]:
import re, numpy as np
from functools import lru_cache
from pythainlp.tokenize import sent_tokenize
from pythainlp.corpus import thai_stopwords
from sklearn.feature_extraction.text import TfidfVectorizer

STOP = set(thai_stopwords())

@lru_cache(maxsize=None)
def tok(text):
    return tuple(w for w in word_tokenize(text, engine="newmm") if w.strip() and w not in STOP)

def tok_list(text):
    return list(tok(text))

def split_sentences(body, engine=None, min_len=None):
    engine, min_len = engine or SENT_ENGINE, min_len or MIN_LEN
    parts = []
    for line in re.split(r"[\r\n]+", body):
        line = line.strip()
        if line:
            parts += sent_tokenize(line, engine="crfcut") if engine == "crfcut" else line.split()
    sents, buf = [], ""
    for p in parts:
        buf = (buf + " " + p).strip() if buf else p.strip()
        if len(buf) >= min_len:
            sents.append(buf); buf = ""
    if buf:                                   # เศษสุดท้ายที่สั้นเกินไป
        if sents: sents[-1] += " " + buf
        else: sents.append(buf)
    return sents

idf_sents = [s for b in idf_df.body for s in split_sentences(b)]
vec = TfidfVectorizer(tokenizer=tok_list, lowercase=False, token_pattern=None).fit(idf_sents)
n_sent = [len(split_sentences(b)) for b in val_df.body]
print(f"ประโยคที่ใช้หา IDF: {len(idf_sents):,} · คำศัพท์: {len(vec.vocabulary_):,} · ประโยค/ข่าว (val) เฉลี่ย {np.mean(n_sent):.1f}")
print("ความยาวบทสรุปจริง (val) เฉลี่ย", round(val_df.summary.str.len().mean()), "ตัวอักษร")

## 5. ให้คะแนนประโยค
- **tfidf:** ผลรวมน้ำหนัก TF-IDF ของคำในประโยค ÷ √(จำนวนคำไม่ซ้ำ) กันไม่ให้ประโยคยาวได้เปรียบเกินไป
- **textrank:** ความคล้ายระหว่างประโยค = cosine ของเวกเตอร์ TF-IDF → กราฟถ่วงน้ำหนัก → PageRank (damping 0.85)
- **lead:** ประโยคต้น ๆ ได้คะแนนสูงกว่า (baseline ที่ในข่าวมักแข็งแรงมาก)

In [ ]:
def pagerank(sim, d=0.85, iters=100, tol=1e-6):
    n = len(sim)
    col = sim.sum(axis=0); col[col == 0] = 1
    M = sim / col
    r = np.full(n, 1 / n)
    for _ in range(iters):
        new = (1 - d) / n + d * M @ r
        if np.abs(new - r).sum() < tol:
            r = new; break
        r = new
    return r

def score_sentences(sents, method):
    n = len(sents)
    if method == "lead":
        return np.arange(n, 0, -1, dtype=float)
    X = vec.transform(sents)
    if method == "tfidf":
        nnz = np.maximum(np.asarray((X > 0).sum(axis=1)).ravel(), 1)
        return np.asarray(X.sum(axis=1)).ravel() / np.sqrt(nnz)
    if method == "textrank":
        if n == 1: return np.ones(1)
        sim = (X @ X.T).toarray(); np.fill_diagonal(sim, 0)
        return pagerank(sim)
    raise ValueError(method)

def summarize_one(body, method, k):
    sents = split_sentences(body)
    sc = score_sentences(sents, method)
    idx = sorted(np.argsort(-sc)[:k])          # เรียงกลับตามลำดับในข่าว
    return " ".join(sents[i] for i in idx)

def summarize(bodies, method, k):
    return [summarize_one(b, method, k) for b in bodies]

## 6. เลือกวิธีและ k จาก validation

In [ ]:
rows = []
for m in METHODS:
    for k in K_GRID:
        rows.append({"method": m, "k": k, **thai_rouge(summarize(val_df.body, m, k), val_df.summary)})
grid = pd.DataFrame(rows).sort_values("rougeL", ascending=False).reset_index(drop=True)
best = grid.iloc[0]
BEST_METHOD, BEST_K = best.method, int(best.k)
print(f"ดีที่สุดบน val: {BEST_METHOD}, k={BEST_K}")
grid

## 7. วัดผลบน test (ส่งตารางนี้ให้คนที่ 7)
รายงานทั้งวิธีที่เลือกและ Lead-k เพื่อให้เห็นว่า TextRank/TF-IDF ดีกว่า baseline แค่ไหน

In [ ]:
import json
preds = summarize(test_df.body, BEST_METHOD, BEST_K)
refs = test_df.summary.tolist()
scores = thai_rouge(preds, refs)

baseline = {f"lead{BEST_K}": thai_rouge(summarize(test_df.body, "lead", BEST_K), refs)}
per_method = {m: thai_rouge(summarize(test_df.body, m, BEST_K), refs) for m in METHODS}

result = {"model": f"extractive-{BEST_METHOD}", **scores, "k": BEST_K, "sent_engine": SENT_ENGINE,
          "idf_size": len(idf_df), "val_size": len(val_df), "test_size": len(test_df),
          "baseline_lead": baseline, "all_methods_on_test": per_method}
print(json.dumps(result, ensure_ascii=False, indent=2))
pd.DataFrame(per_method).T

### บทสรุปทั้งหมด (ส่งให้คนที่ 6 เช็กชื่อเฉพาะ และคนที่ 7 วิเคราะห์ข้อผิดพลาด)

In [ ]:
out = test_df.assign(pred=preds)
out["rougeL"] = [scorer.score(r, p)["rougeL"].fmeasure for p, r in zip(preds, refs)]
out.to_csv(f"predictions_{OUTPUT_NAME}.csv", index=False)
json.dump(result, open(f"result_{OUTPUT_NAME}.json", "w"), ensure_ascii=False, indent=2)
pd.set_option("display.max_colwidth", 200)
print("บทสรุปที่ ROUGE-L ต่ำสุด 5 ข่าว:")
out.sort_values("rougeL")[["summary", "pred", "rougeL"]].head()

### วิเคราะห์เร็ว ๆ: บทสรุป extractive ยาวเกินจริงไหม?
Extractive เลือกทั้งประโยค จึงมักยาวกว่าบทสรุปที่คนเขียน ซึ่งกดคะแนน precision ลองดูความยาวเทียบกัน

In [ ]:
print("ความยาวเฉลี่ย (ตัวอักษร) — บทสรุปจริง:", round(out.summary.str.len().mean()),
      "| extractive:", round(out.pred.str.len().mean()))

## 8. ลองสรุปข่าวใหม่

In [ ]:
news = """กรมอุตุนิยมวิทยาประกาศเตือนพายุฤดูร้อนบริเวณภาคเหนือและภาคตะวันออกเฉียงเหนือ ระหว่างวันที่ 10-12 นี้
จะมีพายุฝนฟ้าคะนอง ลมกระโชกแรง และลูกเห็บตกบางพื้นที่ ขอให้ประชาชนหลีกเลี่ยงการอยู่ในที่โล่งแจ้ง
และใต้ต้นไม้ใหญ่ ส่วนเกษตรกรควรเตรียมป้องกันความเสียหายของผลผลิต"""
summarize_one(news, BEST_METHOD, BEST_K)